## 训练环境与项目日志初始化

In [ ]:
import numpy as np
import json  # 导入json模块
import logging

from data import plot_data, format_dict, make_dirs, TrainingHistory, load_checkpoint
from controller import PIDController, TD3Controller

train_model = True  # 是否训练模型
controller_type = "PID"  # 控制器类型：TD3 或 PPO
history: TrainingHistory = None

In [ ]:
from datetime import datetime
import os
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
project_name = input("请输入加载/创建项目的名称 (父目录: .\\savedata) ").strip()# 创建保存模型的基础目录
project_path, ckpt_dir, plt_dir = make_dirs(project_name)
file_path = os.path.join(project_path, f'training_{current_time}.log')
project_path, ckpt_dir, plot_path = make_dirs(project_name)
if train_model:
    logging.basicConfig(filename=file_path, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # 设置日志格式
logging.info("## 当前时间: %s", datetime.now())
logging.info("项目保存目录: %s", project_path)
logging.info("日志文件: %s", file_path)
logging.info("训练模式: %s", train_model)
print(f"日志文件: {file_path}")

In [ ]:
import numpy as np
# 计算固有频率
m = 1.0    # 电磁吸振器质量
M = 15  # 待减振对象质量
k_m = 30_000  # 电磁吸振器刚度
k_M = 300_000  # 平台刚度
k_f = 100 # * TD3_PARAMS['action_bound']  # 电—力常数 N/A
# k_E = 0.0  # 作动器反电动势系数
# L = 0.0045  # 线圈的电感
# R_m = 5.0  # 线圈的电阻
c_m = 0.0001 # 1.0  # 电磁吸振器阻尼
c_M = 0.001 # 5.0  # 平台阻尼

f1 = np.sqrt(k_m / m) / (2 * np.pi)  # 电磁吸振器的固有频率
f2 = np.sqrt(k_M / M) / (2 * np.pi)  # 待减振对象的固有频率
print(f"电磁吸振器固有频率: {f1:.2f} Hz")
print(f"待减振对象固有频率: {f2:.2f} Hz")

## 训练参数

In [ ]:
from fx import zero, tolerance_smooth_reward

ENV_PARAMS = {
    'Ts': 0.001,  # 环境时间步长
    'T': 1.0,     # 每回合总时间
    'state0': np.array([0.0, 0.0, 0.0, 0.006, 0.0, 0.0]),  # 初始状态
    'obs_indices': [0, 2, 3, 5],  # 观测状态索引
    'x1_limit': 0.03,  # 状态 x1 的限制
    'use_dt_noise': False,  # 是否使用时间步长噪声
    'dt_noise_std': 0.1,  # 时间步长噪声标准差
    'delay_enabled': False,  # 是否启用动作延迟
    'delay_mean_steps': 0,  # 延迟均值步数
    'delay_std_steps': 0,   # 延迟标准差步数
    'include_dt_in_obs': False,  # 是否在观测中包含时间步长
    'include_delay_in_obs': False,  # 是否在观测中包含延迟
    'z_func': zero,  # 状态惩罚函数
    'r_func': tolerance_smooth_reward(1e-3),  # 奖励函数
    'f_func': zero,  # 外部激励函数
    # 'f_func': sin_wave(15000, 30*2*np.pi),  # 外部激励函数
}

DELAY_ENV_PARAMS = ENV_PARAMS.copy()
DELAY_ENV_PARAMS.update({'include_delay_in_obs': True, 'delay_enabled': True, 'delay_mean_steps': 1, 'delay_std_steps': 0})
DELAY_MLP_ENV_PARAMS = DELAY_ENV_PARAMS.copy()
DELAY_GRUA_ENV_PARAMS = DELAY_ENV_PARAMS.copy()
DELAY_MLP_ENV_PARAMS.update({'include_delay_in_obs': False})
DELAY_GRUA_ENV_PARAMS.update({'include_delay_in_obs': True})

PID_PARAMS = {
    'kp': -2300.0,  # 比例增益
    'ki': -120000.0,    # 积分增益
    'kd': -8.0,   # 微分增益
    'target': 0.0,  # 目标值
    'target_idx': 2,  # 目标状态索引
    'dt': ENV_PARAMS['Ts'],  # 时间步长
}

MLP_TD3_PARAMS = {
    # 创建神经网络的参数
    'arch': 'mlp',  # 网络架构类型
    'norm': False,      # 是否使用归一化
    'simple_nn': False,  # 是否使用简单神经网络
    'state_dim': len(ENV_PARAMS['obs_indices']) + int(ENV_PARAMS['include_dt_in_obs']) + int(ENV_PARAMS['include_delay_in_obs']),  # 状态维度
    'action_dim': 1,  # 动作维度
    'action_bound': 5,  # 动作边界
    'gru_hidden': 64,  # GRU 隐藏层大小
    'gru_layers': 1,   # GRU 层数
    'hidden_dim': 64,  # 全连接层隐藏维度
    'seq_len': 36,     # 预测前序列长度
    'fc_seq_len': 4,   # 预测器预测序列长度
    # 优化器参数
    'actor_lr': 5e-06,  # Actor 学习率
    'critic_lr': 1e-05,  # Critic 学习率
    'clip_grad': 1.0,  # 梯度裁剪值
    'tau': 0.002,      # 软更新参数 SEQ取0.001
    # TD3 算法参数
    'gamma': 0.99,     # 折扣因子
    'policy_noise': 0.2,  # 策略噪声标准差
    'noise_clip': 0.5,   # 噪声裁剪值
    'policy_freq': 3,   # 策略延迟频率 
}

DELAY_MLP_TD3_PARAMS = MLP_TD3_PARAMS.copy()
DELAY_MLP_TD3_PARAMS.update({'state_dim': len(DELAY_MLP_ENV_PARAMS['obs_indices']) + int(DELAY_MLP_ENV_PARAMS['include_dt_in_obs']) + int(DELAY_MLP_ENV_PARAMS['include_delay_in_obs'])})

GRUA_TD3_PARAMS = {
    # 创建神经网络的参数
    'arch': 'seq',  # 网络架构类型
    'norm': False,      # 是否使用归一化
    'simple_nn': False,  # 是否使用简单神经网络
    'state_dim': len(DELAY_GRUA_ENV_PARAMS['obs_indices']) + int(DELAY_GRUA_ENV_PARAMS['include_dt_in_obs']) + int(DELAY_GRUA_ENV_PARAMS['include_delay_in_obs']),  # 状态维度
    'action_dim': 1,  # 动作维度
    'action_bound': 5,  # 动作边界
    'gru_hidden': 64,  # GRU 隐藏层大小
    'gru_layers': 1,   # GRU 层数
    'hidden_dim': 128,  # 全连接层隐藏维度
    'seq_len': 36,     # 预测前序列长度
    'fc_seq_len': 4,   # 预测器预测序列长度
    # 优化器参数
    'actor_lr': 5e-06,  # Actor 学习率
    'critic_lr': 1e-05,  # Critic 学习率
    'clip_grad': 1.0,  # 梯度裁剪值
    'tau': 0.001,      # 软更新参数 SEQ取0.001
    # TD3 算法参数
    'gamma': 0.99,     # 折扣因子
    'policy_noise': 0.2,  # 策略噪声标准差
    'noise_clip': 0.5,   # 噪声裁剪值
    'policy_freq': 3,   # 策略延迟频率 
}

logging.info("环境参数:\n%s", json.dumps(format_dict(ENV_PARAMS), indent=4, ensure_ascii=False))
logging.info("GRUA 环境参数:\n%s", json.dumps(format_dict(DELAY_ENV_PARAMS), indent=4, ensure_ascii=False))  
logging.info("PID 参数:\n%s", json.dumps(PID_PARAMS, indent=4, ensure_ascii=False))
logging.info("MLP TD3 参数:\n%s", json.dumps(MLP_TD3_PARAMS, indent=4, ensure_ascii=False))
logging.info("GRUA TD3 参数:\n%s", json.dumps(GRUA_TD3_PARAMS, indent=4, ensure_ascii=False))

## 控制对比分析图

In [ ]:
# 创建控制器
pid_Controller = PIDController(**PID_PARAMS)
GRUA_TD3Controller = TD3Controller(**GRUA_TD3_PARAMS)
model_paths= ["E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习\GRUA-TD3\savedata\SS结题报告对比\S8_0317下午无延迟_0318_122157_ep465ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习/GRUA-TD3/savedata/SS结题报告对比/S8_0321下午-1延迟_0408_160251_ep60ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习\GRUA-TD3\savedata\SS结题报告对比\S8_0323下午-2延迟_0323_155619_ep120ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习\GRUA-TD3\savedata\SS结题报告对比\S8_0323下午-3延迟_0323_173858_ep100ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习/GRUA-TD3/savedata/SS结题报告对比/S8_0323下午-4延迟_0323_183711_ep60ckpt.pth,",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习\GRUA-TD3\savedata\SS结题报告对比\S8_0324下午-4-1延迟_0324_153934_ep100ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习/GRUA-TD3/savedata/SS结题报告对比/S10_0327中午MLP无延迟_0328_002325_ep900ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习\GRUA-TD3\savedata\SS结题报告对比\S10_0328中午MLP-1延迟_0329_194306_ep500ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习/GRUA-TD3/savedata/SS结题报告对比/S10_0331上午MLP-2延迟_0331_132010_ep620ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习\GRUA-TD3\savedata\SS结题报告对比\S10_0331下午MLP-3延迟_0408_151329_ep795ckpt.pth",
              "E:\\100_Study\\130_SubjectLearn\\1310_DySysIdCtrl\电磁式阻尼器深度强化学习\GRUA-TD3\savedata\SS结题报告对比\S10_0401下午MLP-3-1延迟_0401_191717_ep800ckpt.pth"
              ]
GRUA_TD3_Controller = TD3Controller(**GRUA_TD3_PARAMS)
payload = load_checkpoint(model_paths[0])
history = TrainingHistory.from_dict(payload.get("history", None))
grua_td3_reward = history.as_numpy(keys='reward_history')
GRUA_TD3_Controller.load_state(payload["agent"])
cs_MLP_TD3_PARAMS = MLP_TD3_PARAMS.copy()
cs_MLP_TD3_PARAMS.update({'hidden_dim': 128})
MLP_TD3_Controller = TD3Controller(**cs_MLP_TD3_PARAMS)
payload = load_checkpoint(model_paths[6])
history = TrainingHistory.from_dict(payload.get("history", None))
mlp_td3_reward = history.as_numpy(keys='reward_history')
MLP_TD3_Controller.load_state(payload["agent"])

Delay_GRUA_TD3_Controller = TD3Controller(**GRUA_TD3_PARAMS)
payload = load_checkpoint(model_paths[5])
delay_grua_td3_reward = history.as_numpy(keys='reward_history')
Delay_GRUA_TD3_Controller.load_state(payload["agent"])
Delay_MLP_TD3_Controller = TD3Controller(**DELAY_MLP_TD3_PARAMS)
payload = load_checkpoint(model_paths[10])
history = TrainingHistory.from_dict(payload.get("history", None))
delay_mlp_td3_reward = history.as_numpy(keys='reward_history')
Delay_MLP_TD3_Controller.load_state(payload["agent"])
# GRUA_TD3_Controller0 = TD3Controller(**GRUA_TD3_PARAMS)
# payload = load_checkpoint(model_paths[0])
# GRUA_TD3_Controller0.load_state(payload["agent"])
# GRUA_TD3_Controller1 = TD3Controller(**GRUA_TD3_PARAMS)
# payload = load_checkpoint(model_paths[1])
# GRUA_TD3_Controller1.load_state(payload["agent"])
# GRUA_TD3_Controller2 = TD3Controller(**GRUA_TD3_PARAMS)
# payload = load_checkpoint(model_paths[2])
# GRUA_TD3_Controller2.load_state(payload["agent"])
# GRUA_TD3_Controller3 = TD3Controller(**GRUA_TD3_PARAMS)
# payload = load_checkpoint(model_paths[3])
# GRUA_TD3_Controller3.load_state(payload["agent"])
# GRUA_TD3_Controller4 = TD3Controller(**GRUA_TD3_PARAMS)
# payload = load_checkpoint(model_paths[4])
# GRUA_TD3_Controller4.load_state(payload["agent"])
# GRUA_TD3_Controller41 = TD3Controller(**GRUA_TD3_PARAMS)
# payload = load_checkpoint(model_paths[5])
# GRUA_TD3_Controller41.load_state(payload["agent"])
# MLP_TD3_Controller0 = TD3Controller(**MLP_TD3_PARAMS)
# 绘制训练过程中的奖励曲线
plot_data(figsize=(10, 6),y_values=mlp_td3_reward, legends=[('回合奖励',)], plot_title='MLP-TD3训练奖励趋势', legend_loc='upper right', save_path=project_path)
plot_data(figsize=(10, 6),y_values=grua_td3_reward, legends=[('回合奖励',)], plot_title='GRUA-TD3训练奖励趋势', legend_loc='upper right', save_path=project_path)
plot_data(figsize=(10, 6),y_values=delay_mlp_td3_reward, legends=[('回合奖励',)], plot_title='延迟环境下MLP-TD3训练奖励趋势', legend_loc='upper right', save_path=project_path)
plot_data(figsize=(10, 6),y_values=delay_grua_td3_reward, legends=[('回合奖励',)], plot_title='延迟环境下GRUA-TD3训练奖励趋势', legend_loc='upper right', save_path=project_path)

In [ ]:
# 创建环境
from env import build_env
env = build_env(ENV_PARAMS)
delay_env = build_env(DELAY_ENV_PARAMS)
delay_mlp_env = build_env(DELAY_MLP_ENV_PARAMS)

# 无控制
nc_recorder = env.run_episode(controller=None, state0=ENV_PARAMS['state0'], z_func=ENV_PARAMS['z_func'], f_func=ENV_PARAMS['f_func'])
nc_x_values=nc_recorder.as_numpy(keys='time_history').reshape(-1, 1)[:,[0,0,0,0,0,0]]
nc_y_values=nc_recorder.as_numpy(keys='state_history')[:,[0,1,2,3,4,5]]

# PID控制
pid_recorder = env.run_episode(controller=pid_Controller, state0=ENV_PARAMS['state0'], z_func=ENV_PARAMS['z_func'], f_func=ENV_PARAMS['f_func'])
pid_x_values=pid_recorder.as_numpy(keys='time_history').reshape(-1, 1)
pid_y_values=pid_recorder.as_numpy(keys='state_history')[:,[0,1,2,3,4,5]]
plot_data(x_values=pid_x_values, y_values=np.concatenate((nc_y_values[:,[0,1,2,3,4,5]], pid_y_values), axis=1), 
            # legends=[('无控制','PID控制',),('无控制','PID控制'),('无控制','PID控制',),('无控制','PID控制',),('无控制','PID控制',),('无控制','PID控制',)], 
            total_label=['无控制', 'PID控制'], legend_loc='upper right',
            sub_shape=(3, 2), sub_group=[(0,6), (3,9), (1,7), (4,10), (2,8), (5,11)],
            subplot_titles=['吸振器位移', '主结构位移', '吸振器速度', '主结构速度', '吸振器加速度', '主结构加速度'],
            xlim=(0,1),
            plot_title='PID控制器响应对比图', save_path=plot_path, show=True)

# TD3控制
td3_recorder = env.run_episode(controller=MLP_TD3_Controller, state0=ENV_PARAMS['state0'], z_func=ENV_PARAMS['z_func'], f_func=ENV_PARAMS['f_func'])
td3_x_values=td3_recorder.as_numpy(keys='time_history').reshape(-1, 1)
td3_y_values=td3_recorder.as_numpy(keys='state_history')[:,[0,1,2,3,4,5]]
grua_td3_recorder = delay_env.run_episode(controller=GRUA_TD3_Controller, state0=DELAY_ENV_PARAMS['state0'], z_func=DELAY_ENV_PARAMS['z_func'], f_func=DELAY_ENV_PARAMS['f_func'])
grua_td3_x_values=grua_td3_recorder.as_numpy(keys='time_history').reshape(-1, 1)
grua_td3_y_values=grua_td3_recorder.as_numpy(keys='state_history')[:,[0,1,2,3,4,5]]
plot_data(x_values=td3_x_values, y_values=np.concatenate((nc_y_values[:,[0,1,2,3,4,5]], td3_y_values, grua_td3_y_values), axis=1), 
            total_label=['无控制', 'MLP_TD3控制', 'GRUA_TD3控制'], legend_loc='upper right',
            sub_shape=(3, 2), sub_group=[(0,6,12), (3,9,15), (1,7,13), (4,10,16), (2,8,14), (5,11,17)],
            subplot_titles=['吸振器位移', '主结构位移', '吸振器速度', '主结构速度', '吸振器加速度', '主结构加速度'],
            xlim=(0,1),
            plot_title='无延迟环境下TD3控制器响应对比图', save_path=plot_path, show=True)

# 延迟环境下的TD3控制
delay_nc_recorder = delay_env.run_episode(controller=None, state0=DELAY_ENV_PARAMS['state0'], z_func=DELAY_ENV_PARAMS['z_func'], f_func=DELAY_ENV_PARAMS['f_func'])
delay_nc_x_values=delay_nc_recorder.as_numpy(keys='time_history').reshape(-1, 1)[:,[0,0,0,0,0,0]]
delay_nc_y_values=delay_nc_recorder.as_numpy(keys='state_history')[:,[0,1,2,3,4,5]]
delay_td3_recorder = delay_mlp_env.run_episode(controller=Delay_MLP_TD3_Controller, state0=DELAY_MLP_ENV_PARAMS['state0'], z_func=DELAY_MLP_ENV_PARAMS['z_func'], f_func=DELAY_MLP_ENV_PARAMS['f_func'])
delay_td3_x_values=delay_td3_recorder.as_numpy(keys='time_history').reshape(-1, 1)
delay_td3_y_values=delay_td3_recorder.as_numpy(keys='state_history')[:, [0,1,2,3,4,5]]
delay_grua_td3_recorder = delay_env.run_episode(controller=Delay_GRUA_TD3_Controller, state0=DELAY_GRUA_ENV_PARAMS['state0'], z_func=DELAY_GRUA_ENV_PARAMS['z_func'], f_func=DELAY_GRUA_ENV_PARAMS['f_func'])
delay_grua_td3_x_values=delay_grua_td3_recorder.as_numpy(keys='time_history').reshape(-1, 1)
delay_grua_td3_y_values=delay_grua_td3_recorder.as_numpy(keys='state_history')[:, [0,1,2,3,4,5]]
plot_data(x_values=delay_td3_x_values, y_values=np.concatenate((delay_nc_y_values, delay_td3_y_values, delay_grua_td3_y_values), axis=1), 
            total_label=['无控制', 'MLP_TD3控制', 'GRUA_TD3控制'], legend_loc='upper right',
            sub_shape=(3, 2), sub_group=[(0,6,12), (3,9,15), (1,7,13), (4,10,16), (2,8,14), (5,11,17)],
            subplot_titles=['吸振器位移', '主结构位移', '吸振器速度', '主结构速度', '吸振器加速度', '主结构加速度'],
            xlim=(0,1),
            plot_title='延迟-1环境下TD3控制器响应对比图', save_path=plot_path, show=True)